# Notebook Xử Lý Phản Hồi Của Thầy Bùi Cao Vũ (Report 1st Version -> Final Version)

Notebook này giải quyết trọn vẹn các góp ý kỹ thuật trong email review của Thầy Bùi Cao Vũ:
1. **Mục 1 & 2a (Code S2 Refinement & Giải mã con số 3,063):** Chạy kiểm chứng `s2_refine_and_repair.py` trực tiếp từ `articles_v2.jsonl` + `props_lp_raw.jsonl` (không chạm tới `gold_spans.jsonl`), thống kê tách biệt theo tập $\mathcal{D}_{\text{eval}}$ và $\mathcal{D}_{\text{dev}}$ để chứng minh con số `3,063` con trỏ mới gắn (`2,708 eval + 355 dev`) và `3,063` gold spans trên $\mathcal{D}_{\text{eval}}$ (`trong tổng 3,562 spans`) chỉ là trùng hợp ngẫu nhiên khác phạm vi.
2. **Mục 2c (Vai trò của tập $\mathcal{D}_{\text{dev}}$):** Tính toán bảng kết quả Tier 1 trên tập $\mathcal{D}_{\text{dev}}$ (923 điều, 499 gold spans) làm cơ sở thực nghiệm cho việc thiết kế ngưỡng $\tau=0.6$ và luật refine.
3. **Mục 2b & Mục 3 (Ép Abstain = Sai & Tách 100 câu hiệu chỉnh Judge v4):** Sửa logic tính điểm sao cho mọi câu trả lời khớp `ABSTAIN_PAT` đều bị ép `correct = 0` (`cond_acc = 0`), sau đó tính lại Bảng 5 và Bảng 6 (kèm Paired Bootstrap 1,000 lần) trên cả **1,200 câu** và **1,100 câu (`qs[100:]`)**.
4. **Mục 4a (Đầy đủ 8 hệ thống ở Tier 2):** Trích xuất toàn bộ 8 hệ thống (`passage`, `std_raw`, `std`, `lp_raw`, `lp_edge_only`, `lp_noptr`, `lp_out`, `lp`) từ `s6_results.json`.

In [1]:
# CẤU HÌNH ĐƯỜNG DẪN DỮ LIỆU TỰ ĐỘNG (HỖ TRỢ CẢ KAGGLE LẪN LOCAL PC)
import os, sys, glob, json, subprocess
from collections import Counter, defaultdict

is_kaggle = os.path.exists("/kaggle")

if is_kaggle:
    print("=== ĐANG CHẠY TRÊN KAGGLE ===")
    REPO_DIR = "/kaggle/working/dense-x-retrieval-replication"
    if not os.path.exists(REPO_DIR):
        print("Đang clone repository từ GitHub...")
        subprocess.run(["git", "clone", "https://github.com/huynhthinhatran/dense-x-retrieval-replication.git", REPO_DIR], check=True)
    
    def find_file(pattern, name):
        hits = sorted(glob.glob(pattern, recursive=True))
        if not hits:
            raise FileNotFoundError(f"Không tìm thấy {name} với pattern: {pattern}. Hãy chắc chắn đã thêm Notebook Output hoặc Dataset vào Kaggle!")
        return hits[0]

    # Tìm data_v2 (từ output của NB1 lp-prop-q hoặc dataset)
    art_file = find_file("/kaggle/input/**/articles_v2.jsonl", "articles_v2.jsonl")
    DATA_V2_DIR = os.path.dirname(art_file)

    # Tìm file câu hỏi 1,200 câu
    QUESTIONS_EVAL_PATH = find_file("/kaggle/input/**/cond_questions_v2_eval_1200.jsonl", "cond_questions_v2_eval_1200.jsonl")

    # Tìm work_s5 (chứa judge_v4.shard*.jsonl từ output của NB2)
    judge_file = find_file("/kaggle/input/**/judge_v4.shard*.jsonl", "judge_v4.shard*.jsonl")
    WORK_S5_DIR = os.path.dirname(judge_file)

    # File kết quả S6 từ repo
    S6_RESULTS_PATH = os.path.join(REPO_DIR, "02_improvement_lp_prop_vn/s6_evaluation/results/s6_results.json")

else:
    print("=== ĐANG CHẠY TRÊN LOCAL PC ===")
    REPO_DIR = "D:/Documents/Project SEG/dense-x-retrieval-replication"
    DATA_V2_DIR = "D:/Documents/Project SEG/lần chạy mới/results (1)/data_v2"
    WORK_S5_DIR = "D:/Documents/Project SEG/lần chạy mới/results/work_s5"
    QUESTIONS_EVAL_PATH = "D:/Documents/Project SEG/lần chạy mới/results (1)/questions_v2/cond_questions_v2_eval_1200.jsonl"
    S6_RESULTS_PATH = os.path.join(REPO_DIR, "02_improvement_lp_prop_vn/s6_evaluation/results/s6_results.json")

# Thêm đường dẫn scripts vào sys.path
sys.path.insert(0, os.path.join(REPO_DIR, "02_improvement_lp_prop_vn/s2_propositionizer/scripts".replace("/", os.sep)))
sys.path.insert(0, os.path.join(REPO_DIR, "02_improvement_lp_prop_vn/s5_reader_judge/scripts".replace("/", os.sep)))

import s2_refine_and_repair as s2_ref
import s5_reader_judge as s5

print(f"Môi trường: {'Kaggle' if is_kaggle else 'Local PC'}")
print(f"DATA_V2_DIR: {DATA_V2_DIR}")
print(f"WORK_S5_DIR: {WORK_S5_DIR}")
print(f"QUESTIONS_EVAL_PATH: {QUESTIONS_EVAL_PATH}")
print(f"S6_RESULTS_PATH: {S6_RESULTS_PATH}")
print("Đã nạp thành công module s2_refine_and_repair và s5_reader_judge!")


=== ĐANG CHẠY TRÊN LOCAL PC ===
Môi trường: Local PC
DATA_V2_DIR: D:/Documents/Project SEG/lần chạy mới/results (1)/data_v2
WORK_S5_DIR: D:/Documents/Project SEG/lần chạy mới/results/work_s5
QUESTIONS_EVAL_PATH: D:/Documents/Project SEG/lần chạy mới/results (1)/questions_v2/cond_questions_v2_eval_1200.jsonl
S6_RESULTS_PATH: D:/Documents/Project SEG/dense-x-retrieval-replication\02_improvement_lp_prop_vn/s6_evaluation/results/s6_results.json
Đã nạp thành công module s2_refine_and_repair và s5_reader_judge!


## Phần 1: Kiểm chứng S2 Refinement & Giải mã nghi vấn con số `3,063` (Góp ý 1 & 2a, 2c)

In [2]:
# Đọc dữ liệu điều luật, gold_spans và proposition thô
arts = s2_ref.read_jsonl(os.path.join(DATA_V2_DIR, "articles_v2.jsonl"))
by_aid = {a["aid"]: a for a in arts}
eval_aids = {a["aid"] for a in arts if a.get("split") == "eval"}
dev_aids  = {a["aid"] for a in arts if a.get("split") == "dev"}

gold = s2_ref.read_jsonl(os.path.join(DATA_V2_DIR, "gold_spans.jsonl"))
gold_eval = [g for g in gold if g["aid"] in eval_aids]
gold_dev  = [g for g in gold if g["aid"] in dev_aids]

lp_raw = [r for r in s2_ref.read_jsonl(os.path.join(DATA_V2_DIR, "props_lp_raw.jsonl")) if r["aid"] in by_aid]
std_raw = [r for r in s2_ref.read_jsonl(os.path.join(DATA_V2_DIR, "props_std_raw.jsonl")) if r["aid"] in by_aid]

st_eval, st_dev = Counter(), Counter()
lp_edge_only, lp_clean, std_clean = [], [], []
for r in lp_raw:
    art = by_aid[r["aid"]]
    st = st_eval if art.get("split") == "eval" else st_dev
    lp_edge_only.append(s2_ref.refine_lp_record(r, art["article_content"], recover_sents=False))
    lp_clean.append(s2_ref.refine_lp_record(r, art["article_content"], recover_sents=True, stats_counter=st))

for r in std_raw:
    std_clean.append(s2_ref.repair_std_record(r, by_aid[r["aid"]]["article_content"]))

p_raw = {(r["aid"], e["head"], e["tail"]) for r in lp_raw for e in r.get("pointers", [])}
p_lp  = {(r["aid"], e["head"], e["tail"]) for r in lp_clean for e in r.get("pointers", [])}
inv = {(a, t, h) for (a, h, t) in p_raw} & p_lp - p_raw
att = p_lp - p_raw - inv
pru = p_raw - p_lp - {(a, t, h) for (a, h, t) in inv}

print("=== 1. SO SÁNH PHẠM VI CỦA CON SỐ 3,063 ===")
print(f"- Tổng số Gold Spans toàn bộ (7,391 điều): {len(gold):,d}  |  D_eval ({len(eval_aids)} điều): {len(gold_eval):,d}  |  D_dev ({len(dev_aids)} điều): {len(gold_dev):,d}")
print(f"- Số con trỏ Newly Attached (7,391 điều):  {len(att):,d}  |  D_eval ({len(eval_aids)} điều): {sum(1 for a,_,_ in att if a in eval_aids):,d}  |  D_dev ({len(dev_aids)} điều): {sum(1 for a,_,_ in att if a in dev_aids):,d}")
print(f"- Số con trỏ Pruned (7,391 điều):          {len(pru):,d}  |  D_eval: {sum(1 for a,_,_ in pru if a in eval_aids):,d}  |  D_dev: {sum(1 for a,_,_ in pru if a in dev_aids):,d}")
print(f"- Số con trỏ Inverted (7,391 điều):        {len(inv):,d}  |  D_eval: {sum(1 for a,_,_ in inv if a in eval_aids):,d}  |  D_dev: {sum(1 for a,_,_ in inv if a in dev_aids):,d}")
print(f"- Số câu khôi phục Repair (7,391 điều):    {st_eval['recovered_sents'] + st_dev['recovered_sents']:,d}  |  D_eval: {st_eval['recovered_sents']:,d}  |  D_dev: {st_dev['recovered_sents']:,d}")
print("\n=> KẾT LUẬN: Con số 3,063 newly attached pointers là tổng trên TOÀN BỘ 7,391 điều (2,708 trên D_eval + 355 trên D_dev),")
print("   còn 3,063 gold spans là tính riêng trên D_eval (toàn bộ có 3,562 gold spans). Hai con số hoàn toàn khác phạm vi!")

=== 1. SO SÁNH PHẠM VI CỦA CON SỐ 3,063 ===
- Tổng số Gold Spans toàn bộ (7,391 điều): 3,562  |  D_eval (6468 điều): 3,063  |  D_dev (923 điều): 499
- Số con trỏ Newly Attached (7,391 điều):  3,063  |  D_eval (6468 điều): 2,708  |  D_dev (923 điều): 355
- Số con trỏ Pruned (7,391 điều):          12,252  |  D_eval: 10,818  |  D_dev: 1,434
- Số con trỏ Inverted (7,391 điều):        631  |  D_eval: 555  |  D_dev: 76
- Số câu khôi phục Repair (7,391 điều):    316  |  D_eval: 280  |  D_dev: 36

=> KẾT LUẬN: Con số 3,063 newly attached pointers là tổng trên TOÀN BỘ 7,391 điều (2,708 trên D_eval + 355 trên D_dev),
   còn 3,063 gold spans là tính riêng trên D_eval (toàn bộ có 3,562 gold spans). Hai con số hoàn toàn khác phạm vi!


In [3]:
# Đánh giá Tier 1 trên tập D_dev (499 spans) và D_eval (3,063 spans) để báo cáo vai trò của D_dev
def eval_tier1_split(records, gold_list, aid_set, tau=0.6):
    recs = [r for r in records if r["aid"] in aid_set]
    by_a = {r["aid"]: r for r in recs}
    spans_by_aid = defaultdict(list)
    for g in gold_list:
        spans_by_aid[g["aid"]].append(g)
    n = drop = uncond = keep_ptr = 0
    for aid, sps in spans_by_aid.items():
        r = by_a.get(aid)
        if not r: continue
        texts = {p["uid"]: p["text"] for p in r["props"]}
        for sp in sps:
            n += 1
            c_any = any(s2_ref.coverage(sp["condition"], [t]) >= tau for t in texts.values())
            h_any = any(s2_ref.coverage(sp["head"], [t]) >= tau for t in texts.values())
            drop += not c_any
            uncond += h_any and not c_any
            self_c = any(s2_ref.coverage(sp["head"], [t]) >= tau and s2_ref.coverage(sp["condition"], [t]) >= tau for t in texts.values())
            pair = any(s2_ref.coverage(sp["head"], [texts[e["head"]]]) >= tau and s2_ref.coverage(sp["condition"], [texts[e["tail"]]]) >= tau
                       for e in r.get("pointers", []) if e["head"] in texts and e["tail"] in texts)
            keep_ptr += self_c or pair
    return {"n_spans": n, "CondDrop": drop/n*100, "UncondRule": uncond/n*100, "PtrRet": keep_ptr/n*100}

for split_name, g_list, a_set in [("D_dev (923 điều, 499 spans)", gold_dev, dev_aids), ("D_eval (6,468 điều, 3,063 spans)", gold_eval, eval_aids)]:
    print(f"\n=== TIER 1 TRÊN {split_name} ===")
    for name, recs in [("std_raw", std_raw), ("std", std_clean), ("lp_raw", lp_raw), ("lp_edge_only", lp_edge_only), ("lp", lp_clean)]:
        m = eval_tier1_split(recs, g_list, a_set)
        print(f"  {name:14s}: CondDrop = {m['CondDrop']:5.1f}% | UncondRule = {m['UncondRule']:5.1f}% | PtrRet = {m['PtrRet']:5.1f}%")


=== TIER 1 TRÊN D_dev (923 điều, 499 spans) ===


  std_raw       : CondDrop =   4.2% | UncondRule =   2.0% | PtrRet =  89.4%


  std           : CondDrop =   1.2% | UncondRule =   1.2% | PtrRet =  93.2%
  lp_raw        : CondDrop =   7.4% | UncondRule =   6.2% | PtrRet =  86.8%


  lp_edge_only  : CondDrop =   7.4% | UncondRule =   6.2% | PtrRet =  87.4%


  lp            : CondDrop =   3.6% | UncondRule =   3.6% | PtrRet =  92.2%

=== TIER 1 TRÊN D_eval (6,468 điều, 3,063 spans) ===


  std_raw       : CondDrop =   5.9% | UncondRule =   3.2% | PtrRet =  86.3%


  std           : CondDrop =   2.2% | UncondRule =   2.2% | PtrRet =  91.0%


  lp_raw        : CondDrop =   7.7% | UncondRule =   5.3% | PtrRet =  85.9%


  lp_edge_only  : CondDrop =   7.7% | UncondRule =   5.3% | PtrRet =  86.5%


  lp            : CondDrop =   3.3% | UncondRule =   3.3% | PtrRet =  91.7%


## Phần 2: Sửa cách tính điểm `Abstain = Sai` & Tách 100 câu hiệu chỉnh Judge v4 (Góp ý 2b & 3)

Tại `s5_reader_judge.py`, biến `is_abst = bool(ABSTAIN_PAT.search(ans))` được đánh dấu nhưng trước đó `correct` vẫn lấy trực tiếp từ `j["correct"]` của LLM Judge (khiến 429/1,563 câu có chứa cụm từ từ chối vẫn được chấm đúng vì mô hình trả lời nội dung rồi mới ghi thêm câu lưu ý thiếu thông tin). 
Cell dưới đây ép cứng: **hễ `is_abst == True` thì `correct = 0.0` (và kéo theo `cond_acc = 0.0`)**, đồng thời chạy kiểm định Paired Bootstrap 1,000 lần trên cả **1,200 câu** và **1,100 câu (`qs[100:]`)**.

In [4]:
qs = s5.read_jsonl(QUESTIONS_EVAL_PATH)
qmap = {q["qid"]: q for q in qs}
eval_1100_qids = {q["qid"] for q in qs[100:]}

answers = {r["key"]: r for p in glob.glob(os.path.join(WORK_S5_DIR, "reader.shard*.jsonl")) for r in s5.read_jsonl(p)}
mains, conds = {}, {}
for p in sorted(glob.glob(os.path.join(WORK_S5_DIR, "judge_v4.shard*.jsonl"))):
    for r in s5.read_jsonl(p):
        if r["key"].endswith("#c"):
            v = s5.parse_cond(r["raw"])
            if v is not None or r["key"][:-2] not in conds:
                conds[r["key"][:-2]] = v
        else:
            v = s5.parse_main(r["raw"])
            if v is not None or r["key"] not in mains:
                mains[r["key"]] = v

def evaluate_s5(qid_filter=None, force_abstain_wrong=True):
    per = defaultdict(dict)
    for k, r in answers.items():
        qid = r["qid"]
        if qid not in qmap or (qid_filter is not None and qid not in qid_filter):
            continue
        q = qmap[qid]
        ans = r["raw"]
        is_abst = bool(s5.ABSTAIN_PAT.search(ans))
        j = s5.merge_judge(mains.get(k), conds.get(k))
        if j is None:
            continue
        corr = float(j["correct"])
        kept = float(j["condition_kept"])
        overgen = float(j["overgeneralized"])
        if force_abstain_wrong and is_abst:
            corr = 0.0
        cond_acc = float(corr and kept and not (force_abstain_wrong and is_abst))
        p_kept = float(s5.phrase_hit(q["condition"], ans) and not is_abst)
        p_cond_acc = float(corr and p_kept)
        per[(r["system"], str(r["budget"]))][qid] = {
            "correct": corr,
            "condition_kept": kept,
            "cond_acc": cond_acc,
            "overgeneralized": overgen,
            "abstain": float(is_abst),
            "p_kept": p_kept,
            "p_cond_acc": p_cond_acc,
        }
    return per

for title, qf, fix_ab in [
    ("1. BẢNG GỐC TRONG BÀI (1,200 câu, chưa ép Abstain=0)", None, False),
    ("2. BẢNG SAU KHI ÉP ABSTAIN = SAI (1,200 câu - Góp ý 3)", None, True),
    ("3. BẢNG TRÊN 1,100 CÂU NGOÀI TẬP CALIB + ÉP ABSTAIN = SAI (qs[100:] - Góp ý 2b & 3)", eval_1100_qids, True),
]:
    print("\n" + "="*95)
    print(title)
    print("="*95)
    per = evaluate_s5(qf, fix_ab)
    print(f"{'System':10s} | {'Budget':6s} | {'Acc':6s} | {'Kept':6s} | {'Cond-Acc':8s} | {'Over-gen':8s} | {'Abst':6s} || {'Phrase Kept':11s} | {'Phrase Cond-Acc':15s}")
    print("-"*95)
    for B in ["100", "500", "full"]:
        for s in ["passage", "std", "lp_noptr", "lp_out", "lp", "oracle"]:
            if (s, B) not in per: continue
            d = per[(s, B)]; n = len(d)
            m = {k: sum(x[k] for x in d.values())/n*100 for k in ["correct", "condition_kept", "cond_acc", "overgeneralized", "abstain", "p_kept", "p_cond_acc"]}
            print(f"{s:10s} | B{B:5s} | {m['correct']:5.1f}% | {m['condition_kept']:5.1f}% | {m['cond_acc']:7.1f}% | {m['overgeneralized']:7.1f}% | {m['abstain']:5.1f}% || {m['p_kept']:10.1f}% | {m['p_cond_acc']:14.1f}%")
    print("\nPaired Bootstrap 95% CI cho Cond-Acc (lp trừ đối thủ):")
    for B in ["100", "500"]:
        L = per[("lp", B)]
        line = []
        for s in ["passage", "std", "lp_noptr", "lp_out"]:
            d = per[(s, B)]
            ks = sorted(set(L) & set(d))
            dlt, lo, hi = s5.bootstrap_diff([L[k]["cond_acc"] for k in ks], [d[k]["cond_acc"] for k in ks], 1000)
            sig = "*" if not (lo <= 0 <= hi) else ""
            line.append(f"{s}: {dlt*100:+.1f} [{lo*100:+.1f}, {hi*100:+.1f}]{sig}")
        print(f"  B{B}: " + "  ||  ".join(line))


1. BẢNG GỐC TRONG BÀI (1,200 câu, chưa ép Abstain=0)


System     | Budget | Acc    | Kept   | Cond-Acc | Over-gen | Abst   || Phrase Kept | Phrase Cond-Acc
-----------------------------------------------------------------------------------------------
passage    | B100   |  54.8% |  53.4% |    46.0% |     9.1% |  29.2% ||       33.6% |           30.6%
std        | B100   |  73.5% |  72.1% |    63.6% |     8.7% |  15.6% ||       47.9% |           44.6%
lp_noptr   | B100   |  72.8% |  68.6% |    60.4% |    11.1% |  16.1% ||       44.3% |           40.8%
lp_out     | B100   |  74.9% |  71.1% |    62.7% |     8.4% |  15.8% ||       47.5% |           44.0%
lp         | B100   |  70.9% |  66.7% |    58.7% |     9.5% |  17.8% ||       44.3% |           40.7%
passage    | B500   |  79.1% |  77.6% |    69.2% |     8.8% |   6.3% ||       54.9% |           50.0%
std        | B500   |  78.8% |  77.1% |    67.1% |     8.8% |   5.8% ||       53.7% |           47.8%
lp_noptr   | B500   |  78.3% |  74.2% |    65.1% |    10.7% |   6.6% ||       51.7% |   

  B100: passage: +12.7 [+9.0, +16.1]*  ||  std: -4.9 [-8.1, -2.1]*  ||  lp_noptr: -1.8 [-4.4, +1.0]  ||  lp_out: -4.0 [-6.2, -1.7]*


  B500: passage: +0.8 [-2.6, +3.9]  ||  std: +2.8 [-0.1, +5.6]  ||  lp_noptr: +4.8 [+2.1, +7.3]*  ||  lp_out: -0.3 [-2.8, +2.1]

2. BẢNG SAU KHI ÉP ABSTAIN = SAI (1,200 câu - Góp ý 3)


System     | Budget | Acc    | Kept   | Cond-Acc | Over-gen | Abst   || Phrase Kept | Phrase Cond-Acc
-----------------------------------------------------------------------------------------------
passage    | B100   |  52.1% |  53.4% |    43.9% |     9.1% |  29.2% ||       33.6% |           30.6%
std        | B100   |  68.6% |  72.1% |    59.5% |     8.7% |  15.6% ||       47.9% |           44.6%
lp_noptr   | B100   |  67.8% |  68.6% |    56.6% |    11.1% |  16.1% ||       44.3% |           40.8%
lp_out     | B100   |  70.1% |  71.1% |    59.1% |     8.4% |  15.8% ||       47.5% |           44.0%
lp         | B100   |  66.5% |  66.7% |    55.8% |     9.5% |  17.8% ||       44.3% |           40.7%
passage    | B500   |  77.1% |  77.6% |    67.4% |     8.8% |   6.3% ||       54.9% |           50.0%
std        | B500   |  76.4% |  77.1% |    65.2% |     8.8% |   5.8% ||       53.7% |           47.8%
lp_noptr   | B500   |  76.0% |  74.2% |    63.6% |    10.7% |   6.6% ||       51.7% |   

  B100: passage: +11.9 [+8.3, +15.3]*  ||  std: -3.7 [-6.9, -0.8]*  ||  lp_noptr: -0.8 [-3.5, +2.0]  ||  lp_out: -3.2 [-5.5, -0.8]*


  B500: passage: +0.1 [-3.0, +3.0]  ||  std: +2.3 [-0.7, +5.2]  ||  lp_noptr: +3.9 [+1.2, +6.5]*  ||  lp_out: -0.8 [-3.2, +1.8]

3. BẢNG TRÊN 1,100 CÂU NGOÀI TẬP CALIB + ÉP ABSTAIN = SAI (qs[100:] - Góp ý 2b & 3)


System     | Budget | Acc    | Kept   | Cond-Acc | Over-gen | Abst   || Phrase Kept | Phrase Cond-Acc
-----------------------------------------------------------------------------------------------
passage    | B100   |  52.1% |  53.5% |    43.9% |     9.0% |  28.5% ||       33.7% |           30.6%
std        | B100   |  68.0% |  71.1% |    58.5% |     9.0% |  15.5% ||       47.4% |           44.0%
lp_noptr   | B100   |  68.3% |  68.1% |    56.6% |    11.1% |  15.5% ||       43.8% |           40.6%
lp_out     | B100   |  70.9% |  70.9% |    59.5% |     8.5% |  15.1% ||       47.4% |           44.1%
lp         | B100   |  67.4% |  66.8% |    56.4% |     9.5% |  17.0% ||       44.7% |           40.9%
passage    | B500   |  77.2% |  77.5% |    67.5% |     8.9% |   6.4% ||       54.7% |           49.7%
std        | B500   |  76.4% |  76.7% |    64.9% |     8.7% |   5.7% ||       52.8% |           47.2%
lp_noptr   | B500   |  75.6% |  73.6% |    62.8% |    11.2% |   6.7% ||       50.5% |   

  B100: passage: +12.5 [+8.7, +16.0]*  ||  std: -2.2 [-5.4, +0.7]  ||  lp_noptr: -0.3 [-3.2, +2.6]  ||  lp_out: -3.2 [-5.5, -0.7]*


  B500: passage: -0.1 [-3.5, +3.1]  ||  std: +2.5 [-0.5, +5.8]  ||  lp_noptr: +4.6 [+1.6, +7.5]*  ||  lp_out: -0.5 [-3.1, +2.0]


## Phần 3: Xuất bảng Tier 2 đầy đủ 8 hệ thống (Góp ý 4a)

Trích xuất toàn bộ 8 hệ thống (`passage`, `std_raw`, `std`, `lp_raw`, `lp_edge_only`, `lp_noptr`, `lp_out`, `lp`) từ `s6_results.json` để đưa vào Bảng 4 (hoặc Phụ lục).

In [5]:
s6 = json.load(open(S6_RESULTS_PATH, encoding="utf-8"))
t2 = s6["tier2"]
order = ["passage", "std_raw", "std", "lp_raw", "lp_edge_only", "lp_noptr", "lp_out", "lp"]
print(f"{'System':14s} | {'R@5 Cond':8s} | {'R@5 Gen':8s} | {'MRR Gen':8s} || {'CCR@100':7s} | {'Orph@100':8s} || {'CCR@500':7s} | {'Orph@500':8s}")
print("-"*95)
for s in order:
    r = t2[s]
    print(f"{s:14s} | {r['Recall@5_cond']*100:7.1f}% | {r['Recall@5_generic']*100:7.1f}% | {r['MRR@5_generic']*100:7.1f}% || {r['CCR@B100']*100:6.1f}% | {r['Orphan@B100']*100:7.1f}% || {r['CCR@B500']*100:6.1f}% | {r['Orphan@B500']*100:7.1f}%")

System         | R@5 Cond | R@5 Gen  | MRR Gen  || CCR@100 | Orph@100 || CCR@500 | Orph@500
-----------------------------------------------------------------------------------------------
passage        |    89.7% |    88.8% |    74.4% ||   51.4% |     7.8% ||   83.3% |     1.4%
std_raw        |    95.2% |    87.6% |    71.5% ||   70.1% |    12.6% ||   83.8% |     9.4%
std            |    96.1% |    87.7% |    71.6% ||   72.5% |    11.2% ||   86.9% |     7.8%
lp_raw         |    94.3% |    86.4% |    69.9% ||   65.2% |    10.8% ||   85.4% |     6.8%
lp_edge_only   |    94.3% |    86.4% |    69.9% ||   66.2% |    10.8% ||   86.3% |     6.6%
lp_noptr       |    95.3% |    86.5% |    70.0% ||   69.1% |    15.7% ||   83.6% |    10.7%
lp_out         |    95.3% |    86.5% |    70.0% ||   74.2% |     9.3% ||   88.6% |     5.9%
lp             |    95.3% |    86.5% |    70.0% ||   69.3% |     9.5% ||   89.2% |     5.2%
